In [2]:
import os
import pandas as pd
import torch
from tqdm import tqdm
from torch_geometric.data import Data
from sklearn.model_selection import train_test_split
def get_initial_data(random_state=42):
    '''
    missing_corrected.csv를 불러오고,
    데이터 스플릿까지 하는 함수
    Args:
        random_state(int, optional): 데이터 스플릿할 때 필요한 랜덤 시드
    '''
    CURDIR = os.getcwd()
    DATA_PATH = os.path.join(CURDIR, 'missing_corrected.csv')
    DATA = pd.read_csv(DATA_PATH)
    DATA.head()

    # 범주형 변수 더미화 시 train/test 간 불일치를 방지하기 위해
    # 스플릿 전에 전체 데이터 기준으로 가능한 범주를 정의하고 CategoricalDtype으로 고정
    for col in DATA.columns:
        cats = list(DATA[col].unique())
        DATA[col] = DATA[col].astype(pd.api.types.CategoricalDtype(categories=cats))

    # validation set, test set이 imputation 설계하는 데 들어가면 안 됨, 일종의 사후판단 정보가 들어갈 수 있기 때문
    y = DATA['REASONb']
    X = DATA.drop('REASONb', axis=1)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

In [3]:
data = get_initial_data()
X_train = data[0]
X_train.shape


(975896, 73)

In [4]:
from sklearn.feature_selection import mutual_info_classif
import pandas as pd
import numpy as np

def best_n_neighbors(X, y, neighbor_candidates=[3, 5, 7, 10, 15, 20]):
    """
    여러 n_neighbors 값을 테스트하여 평균 Mutual Information이 가장 높은 값을 선택한다.
    """
    result = {}

    for k in neighbor_candidates:
        mi = mutual_info_classif(
            X,
            y,
            discrete_features=True,
            n_neighbors=k,
            random_state=42
        )
        result[k] = np.mean(mi)   # 각 k에 대한 MI 평균 저장

    # 가장 높은 MI 평균을 가지는 k 선택
    best_k = max(result, key=result.get)

    return best_k, result


# 사용 예시
'''
from sklearn.feature_selection import mutual_info_classif
nostfips = X_train.drop('STFIPS', axis=1)
stfips = X_train['STFIPS']

mi = mutual_info_classif(nostfips, stfips, discrete_features=True, n_neighbors=3, random_state=42)
mi_series = pd.Series(mi, index=nostfips.columns).sort_values(ascending=False)

best_k, result_dict = best_n_neighbors(nostfips, stfips)
print("Best n_neighbors:", best_k)
print("MI mean by k:", result_dict)
'''


'\nfrom sklearn.feature_selection import mutual_info_classif\nnostfips = X_train.drop(\'STFIPS\', axis=1)\nstfips = X_train[\'STFIPS\']\n\nmi = mutual_info_classif(nostfips, stfips, discrete_features=True, n_neighbors=3, random_state=42)\nmi_series = pd.Series(mi, index=nostfips.columns).sort_values(ascending=False)\n\nbest_k, result_dict = best_n_neighbors(nostfips, stfips)\nprint("Best n_neighbors:", best_k)\nprint("MI mean by k:", result_dict)\n'

In [ ]:
from sklearn.feature_selection import mutual_info_classif
mi_list = {}

for col in X_train.columns:
    print(col)
    x = X_train.drop(col, axis=1)
    y = X_train[col]

    best_k, result_dict = best_n_neighbors(x, y)
    print("Best n_neighbors:", best_k)

    mi = mutual_info_classif(x, y, discrete_features=True, n_neighbors=best_k, random_state=42)

    mi_list[col] = mi
    print(f'{col} finished')


STFIPS
Best n_neighbors: 3
STFIPS finished
EDUC
Best n_neighbors: 3
EDUC finished
MARSTAT
Best n_neighbors: 3
MARSTAT finished
SERVICES
Best n_neighbors: 3
SERVICES finished
DETCRIM
Best n_neighbors: 3
DETCRIM finished
LOS
Best n_neighbors: 3
LOS finished
PSOURCE
Best n_neighbors: 3
PSOURCE finished
NOPRIOR
Best n_neighbors: 3
NOPRIOR finished
ARRESTS
Best n_neighbors: 3
ARRESTS finished
EMPLOY
Best n_neighbors: 3
EMPLOY finished
METHUSE
Best n_neighbors: 3
METHUSE finished
PSYPROB
Best n_neighbors: 3
PSYPROB finished
PREG
Best n_neighbors: 3
PREG finished
GENDER
Best n_neighbors: 3
GENDER finished
VET
Best n_neighbors: 3
VET finished
LIVARAG
Best n_neighbors: 3
LIVARAG finished
DAYWAIT
Best n_neighbors: 3
DAYWAIT finished
SERVICES_D
Best n_neighbors: 3
SERVICES_D finished
EMPLOY_D
Best n_neighbors: 3
EMPLOY_D finished
LIVARAG_D
Best n_neighbors: 3
LIVARAG_D finished
ARRESTS_D
Best n_neighbors: 3
ARRESTS_D finished
DSMCRIT
Best n_neighbors: 3
DSMCRIT finished
AGE
Best n_neighbors: 3
AG

In [5]:
from sklearn.feature_selection import mutual_info_classif
mi_list = {}

for col in X_train.columns:
    print(col)
    x = X_train.drop(col, axis=1)
    y = X_train[col]

    mi = mutual_info_classif(x, y, discrete_features=True, n_neighbors=3, random_state=42)

    mi_list[col] = mi
    print(f'{col} finished')


STFIPS
STFIPS finished
EDUC
EDUC finished
MARSTAT
MARSTAT finished
SERVICES
SERVICES finished
DETCRIM
DETCRIM finished
LOS
LOS finished
PSOURCE
PSOURCE finished
NOPRIOR
NOPRIOR finished
ARRESTS
ARRESTS finished
EMPLOY
EMPLOY finished
METHUSE
METHUSE finished
PSYPROB
PSYPROB finished
PREG
PREG finished
GENDER
GENDER finished
VET
VET finished
LIVARAG
LIVARAG finished
DAYWAIT
DAYWAIT finished
SERVICES_D
SERVICES_D finished
EMPLOY_D
EMPLOY_D finished
LIVARAG_D
LIVARAG_D finished
ARRESTS_D
ARRESTS_D finished
DSMCRIT
DSMCRIT finished
AGE
AGE finished
RACE
RACE finished
ETHNIC
ETHNIC finished
DETNLF
DETNLF finished
DETNLF_D
DETNLF_D finished
PRIMINC
PRIMINC finished
SUB1
SUB1 finished
SUB2
SUB2 finished
SUB3
SUB3 finished
SUB1_D
SUB1_D finished
SUB2_D
SUB2_D finished
SUB3_D
SUB3_D finished
ROUTE1
ROUTE1 finished
ROUTE2
ROUTE2 finished
ROUTE3
ROUTE3 finished
FREQ1
FREQ1 finished
FREQ2
FREQ2 finished
FREQ3
FREQ3 finished
FREQ1_D
FREQ1_D finished
FREQ2_D
FREQ2_D finished
FREQ3_D
FREQ3_D finished

In [9]:
mi_dict = {}
for col in X_train.columns:
    x_col = X_train.drop(col, axis=1)
    x_col = x_col.columns
    mi_series = pd.Series(mi_list[col], index=x_col)
    mi_series = mi_series.sort_values(ascending=False)
    mi_dict[col] = mi_series

In [11]:
import pickle
with open("mi_dict.pickle", 'wb') as f:
    pickle.dump(mi_dict, f)

In [ ]:
'''
짧게 핵심만 줄게.

## 시작값 추천

* 변수 개수 `p` 기준 휴리스틱: **k = ceil(log2 p)** 를 시작점으로 잡고, **[3, 20]** 범위로 클리핑.

* p < 30 → k = 2–4
* 50 ≤ p ≤ 200 → k = 5–10
* p ≥ 500 → k = 10–20

## 왜 이렇게?

* 너무 작으면 그래프가 끊김(성능↓), 너무 크면 **완전그래프처럼 비용↑/노이즈↑**.
* `log p` 수준은 **연결성**과 **희소성**의 균형을 주는 보편적 출발점.

## 실제 적용 팁

1. **대칭(top-k mutual)**: 서로가 상대를 top-k로 뽑았을 때만 연결 → 허브 과밀 완화.
2. **연결성 체크**: 컴포넌트가 많으면 k를 소폭↑(예: +2).
3. **그리드 스윕**: {3,5,8,10,15,20} 중에서 **검증 성능**(F1/AUROC/Loss)으로 선택.
4. **가중치 유지**: edge_attr에 MI를 두면 k를 조금 낮춰도 정보 보전.
5. **드롭 급락점**: 각 변수의 MI 시리즈에서 **기울기 급락 지점**을 k로 잡는 데이터-적응 방식도 효과적.

요약: **k = ceil(log2 p)**로 시작해서, **연결성/검증성능/계산비용**을 보며 {±2} 범위에서 빠르게 튜닝하는 걸 권장.
'''

In [12]:
import pickle
with open('mi_dict.pickle', 'rb') as f:
    mi_dict = pickle.load(f)
mi_dict

{'STFIPS': DIVISION    2.101302
 CBSA2020    2.091105
 REGION      1.381283
 PRIMPAY     0.868864
 HLTHINS     0.759915
               ...   
 METHFLG     0.001397
 HALLFLG     0.001151
 BARBFLG     0.000482
 INHFLG      0.000272
 TRNQFLG     0.000169
 Length: 72, dtype: float64,
 'EDUC': STFIPS     0.362169
 LIVARAG    0.306288
 SUB1       0.289132
 ROUTE1     0.286771
 EMPLOY     0.278598
              ...   
 AMPHFLG    0.000222
 METHFLG    0.000208
 OTCFLG     0.000196
 INHFLG     0.000115
 TRNQFLG    0.000035
 Length: 72, dtype: float64,
 'MARSTAT': STFIPS       0.345997
 CBSA2020     0.234939
 DIVISION     0.174774
 LIVARAG_D    0.152420
 REGION       0.144743
                ...   
 METHFLG      0.000068
 BARBFLG      0.000066
 SEDHPFLG     0.000027
 INHFLG       0.000017
 TRNQFLG      0.000007
 Length: 72, dtype: float64,
 'SERVICES': SERVICES_D    1.333141
 STFIPS        0.298998
 CBSA2020      0.295607
 LOS           0.276616
 FREQ1         0.135352
                 ...   
 O

In [ ]:
import torch

def build_edge_index_from_mi_directed(mi_dict, top_k=6, return_edge_attr=False):
    """
    mi_dict: {col_name: pd.Series} (index=다른 변수명, value=MI, 내림차순 정렬)
    top_k: 각 src에서 MI 상위 k개로만 유향 엣지(src->dst) 생성
    return_edge_attr: True면 edge_attr로 MI 가중치 반환
    """
    cols = list(mi_dict.keys())
    col_to_idx = {c: i for i, c in enumerate(cols)}

    src_idx, dst_idx, weights = [], [], []

    for src in cols:
        series = mi_dict[src]

        # 우리 그래프에 존재하는 변수만 남기고, 자기 자신은 제외
        series = series[series.index.isin(cols)]
        if src in series.index:
            series = series.drop(index=src)

        # 상위 k개 선택 (이미 내림차순 정렬되어 있다고 가정)
        top_neighbors = series.head(top_k)

        for dst, w in top_neighbors.items():
            src_idx.append(col_to_idx[src])
            dst_idx.append(col_to_idx[dst])
            if return_edge_attr:
                weights.append(float(w))

    edge_index = torch.tensor([src_idx, dst_idx], dtype=torch.long)

    if return_edge_attr:
        edge_attr = torch.tensor(weights, dtype=torch.float)
        return edge_index, edge_attr

    return edge_index


In [ ]:
edge_index = build_edge_index_from_mi_directed(mi_dict, top_k=7, return_edge_attr=False)

In [ ]:
edge_index.shape

torch.Size([2, 511])

In [ ]:
def mi_edge_index(mi_dict_path, top_k=6, return_edge_attr=False):
    """
    mi_dict: {col_name: pd.Series} (index=다른 변수명, value=MI, 내림차순 정렬)
    top_k: 각 src에서 MI 상위 k개로만 유향 엣지(src->dst) 생성
    return_edge_attr: True면 edge_attr로 MI 가중치 반환
    """

    import pickle
    with open(mi_dict_path, 'rb') as f:
        mi_dict = pickle.load(f)

    cols = list(mi_dict.keys())
    col_to_idx = {c: i for i, c in enumerate(cols)}

    src_idx, dst_idx, weights = [], [], []

    for src in cols:
        series = mi_dict[src]

        # 우리 그래프에 존재하는 변수만 남기고, 자기 자신은 제외
        series = series[series.index.isin(cols)]
        if src in series.index:
            series = series.drop(index=src)

        # 상위 k개 선택 (이미 내림차순 정렬되어 있다고 가정)
        top_neighbors = series.head(top_k)

        for dst, w in top_neighbors.items():
            src_idx.append(col_to_idx[src])
            dst_idx.append(col_to_idx[dst])
            if return_edge_attr:
                weights.append(float(w))

    edge_index = torch.tensor([src_idx, dst_idx], dtype=torch.long)

    if return_edge_attr:
        edge_attr = torch.tensor(weights, dtype=torch.float)
        return edge_index, edge_attr

    return edge_index

def mi_edge_index_batched(batch_size, mi_dict_path, top_k=6, return_edge_attr=False):
    single = mi_edge_index(mi_dict_path=mi_dict_path, top_k=top_k, return_edge_attr=return_edge_attr)
    
    batch_list = [single for _ in range(batch_size)]

    return torch.concatenate(tensors=batch_list, dim=1)

In [ ]:
edge_index_batched = mi_edge_index_batched(batch_size=32, mi_dict_path='mi_dict.pickle')